<a href="https://colab.research.google.com/github/yuffiezhaohk/yuffiezhao/blob/main/part2-sparseshocks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y jax jaxlib

In [ ]:
!pip install tensorflow-probability==0.24.0 choice-learn

In [ ]:
# %% [Cell 1] Global Configuration

import unittest
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp
from scipy.optimize import minimize_scalar
from scipy.stats import norm
import tqdm as tqdm_module  # Rename module to avoid conflict
from choice_learn.data import ChoiceDataset
import gc
import logging

tfd = tfp.distributions

# Configure Environment
tf.keras.backend.set_floatx('float64')
logging.getLogger().setLevel(logging.ERROR)


import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_probability as tfp
from choice_learn.data import ChoiceDataset
import tqdm # 进度条工具

tfd = tfp.distributions

In [ ]:
# %% [Cell 2] Data Generation

class SimulationExperiment:
    """
    Simulates market data for BLP experiments including Endogeneity and Sparsity.
    """
    def __init__(self, n_markets: int, n_consumers: int, n_items: int,
                 dgp_type: int, seed: int):
        self.T = n_markets
        self.N = n_consumers
        self.J_real = n_items
        self.J_total = n_items + 1  # Index 0 is the outside option
        self.dgp_type = dgp_type

        self.true_params = {
            'beta_p': -1.0,
            'beta_w': 0.5,
            'sigma': 1.5,
            'xi_bar': -1.0
        }

        tf.random.set_seed(seed)
        np.random.seed(seed)

    def _generate_structural_error(self) -> tf.Tensor:
        if self.dgp_type in [1, 2]:  # Sparse settings
            n_nonzero = int(0.4 * self.J_real)
            eta_vals = [1.0 if (j + 1) % 2 != 0 else -1.0 for j in range(n_nonzero)]
            eta_vals += [0.0] * (self.J_real - n_nonzero)
            eta = tf.tile(tf.constant([eta_vals], dtype=tf.float64), [self.T, 1])
        else:  # Non-sparse settings
            eta = tf.random.normal((self.T, self.J_real), 0.0, 1.0 / 3.0, dtype=tf.float64)
        return eta

    def _calculate_price_endogeneity(self, eta: tf.Tensor) -> tf.Tensor:
        alpha = tf.zeros_like(eta)
        if self.dgp_type in [1, 3]:  # Exogenous
            return alpha

        if self.dgp_type == 2:  # Sparse Endogenous
            alpha = tf.where(tf.abs(eta - 1.0) < 1e-5, 0.3, alpha)
            alpha = tf.where(tf.abs(eta + 1.0) < 1e-5, -0.3, alpha)
        elif self.dgp_type == 4:  # Non-Sparse Endogenous
            alpha = tf.where(eta >= (1.0 / 3.0), 0.3, alpha)
            alpha = tf.where(eta <= (-1.0 / 3.0), -0.3, alpha)
        return alpha

    def run(self) -> ChoiceDataset:
        w_real = tf.cast(tfp.distributions.Uniform(1.0, 2.0).sample((self.T, self.J_real)), tf.float64)
        eta_real = self._generate_structural_error()
        xi_real = self.true_params['xi_bar'] + eta_real
        u_shock = tf.random.normal((self.T, self.J_real), 0.0, 0.7, dtype=tf.float64)
        alpha = self._calculate_price_endogeneity(eta_real)
        p_real = alpha + 0.3 * w_real + u_shock

        zeros_col = tf.zeros((self.T, 1), dtype=tf.float64)
        p_all = tf.concat([zeros_col, p_real], axis=1)
        w_all = tf.concat([zeros_col, w_real], axis=1)
        xi_all = tf.concat([zeros_col, xi_real], axis=1)

        beta_pi = tf.random.normal((self.T, self.N, 1), self.true_params['beta_p'], self.true_params['sigma'], dtype=tf.float64)
        p_exp = tf.expand_dims(p_all, 1)
        w_exp = tf.expand_dims(w_all, 1)
        xi_exp = tf.expand_dims(xi_all, 1)

        V_ijt = (beta_pi * p_exp) + (self.true_params['beta_w'] * w_exp) + xi_exp
        epsilon = tf.cast(tfp.distributions.Gumbel(0.0, 1.0).sample((self.T, self.N, self.J_total)), tf.float64)
        choices = tf.argmax(V_ijt + epsilon, axis=2, output_type=tf.int32)

        return self._package_dataset(p_exp, w_exp, choices, u_shock, eta_real)

    def _package_dataset(self, p_exp, w_exp, choices, u_shock, eta_real) -> ChoiceDataset:
        total_obs = self.T * self.N
        choices_flat = tf.reshape(choices, [-1]).numpy()
        p_flat = tf.reshape(tf.tile(p_exp, [1, self.N, 1]), [total_obs, self.J_total, 1])
        w_flat = tf.reshape(tf.tile(w_exp, [1, self.N, 1]), [total_obs, self.J_total, 1])
        items_features = tf.concat([p_flat, w_flat], axis=2).numpy()

        dataset = ChoiceDataset(
            items_features_by_choice=items_features,
            choices=choices_flat,
            available_items_by_choice=np.ones((total_obs, self.J_total), int),
            items_features_by_choice_names=["price", "weight"]
        )
        dataset.true_u_shock = u_shock.numpy()
        dataset.true_eta = eta_real.numpy()
        return dataset

In [ ]:
# %% [Cell 2] BLP

class BLPEstimator:
    """
    Implements the Berry-Levinsohn-Pakes (BLP) estimator using GMM.

    Attributes:
        iv_strategy (str): 'cost_iv' (strong) or 'no_cost_iv' (weak).
    """

    def __init__(self, dataset: ChoiceDataset, T: int, J_real: int, iv_strategy: str):
        self.T = T
        self.J_real = J_real
        self.J_total = J_real + 1

        self.p, self.w = self._extract_features(dataset)
        self.s_obs, self.s_outside = self._compute_shares(dataset)
        self.Z, self.X = self._construct_instruments_and_regressors(dataset, iv_strategy)

        # Pre-compute GMM weight matrix parts
        self.inv_ZZ = np.linalg.pinv(self.Z.T @ self.Z)

        # Fixed random draws for integration
        self.nu_i = tf.random.normal((1, 500, 1), dtype=tf.float64)

        # Storage for results
        self.last_beta = None
        self.last_xi = None

    def _extract_features(self, dataset):
        """Extracts price and weight matrices from ChoiceDataset."""
        feat_array = dataset.items_features_by_choice[0]
        raw_feats = np.array(feat_array).reshape(self.T, -1, self.J_total, 2)
        # Market features are constant across consumers, take 0th index
        # Slice [:, 1:, :] to exclude outside option (index 0)
        p = raw_feats[:, 0, 1:, 0].astype(np.float64)
        w = raw_feats[:, 0, 1:, 1].astype(np.float64)
        return p, w

    def _compute_shares(self, dataset):
        """Computes observed market shares from consumer choices."""
        choices_mat = dataset.choices.reshape(self.T, -1)
        counts = np.array([np.bincount(choices_mat[t], minlength=self.J_total) for t in range(self.T)])
        total_N = counts.sum(axis=1, keepdims=True)

        s_outside = np.maximum(counts[:, 0:1] / total_N, 1e-8)
        s_inside = np.maximum(counts[:, 1:] / total_N, 1e-8)
        return s_inside, s_outside

    def _construct_instruments_and_regressors(self, dataset, iv_strategy):
        """Constructs Regressor matrix X and Instrument matrix Z."""
        ones = np.ones((self.T, self.J_real))
        # X = [1, p, w]
        X = np.stack([ones, self.p, self.w], axis=2).reshape(-1, 3)

        # Z construction
        if iv_strategy == 'cost_iv':
            u = dataset.true_u_shock
            Z_stack = [ones, self.w, self.w**2, u, u**2]
        else:
            Z_stack = [ones, self.w, self.w**2, self.w**3, self.w**4]

        Z = np.stack(Z_stack, axis=2).reshape(-1, 5)
        return Z, X

    def solve_delta(self, sigma: float) -> np.ndarray:
        """
        Solves for mean utility (delta) using Contraction Mapping.
        """
        # Berry Inversion initialization
        delta = tf.convert_to_tensor(np.log(self.s_obs) - np.log(self.s_outside), dtype=tf.float64)
        p_tf = tf.constant(self.p, dtype=tf.float64)

        for _ in range(2000):
            # mu = sigma * nu * p
            mu = sigma * self.nu_i * tf.expand_dims(p_tf, 1)

            exp_val = tf.exp(tf.expand_dims(delta, 1) + mu)
            denom = 1.0 + tf.reduce_sum(exp_val, axis=2, keepdims=True)
            s_pred = tf.reduce_mean(exp_val / denom, axis=1)

            # Update step
            delta_new = delta + np.log(self.s_obs) - tf.math.log(s_pred + 1e-9)

            if tf.reduce_max(tf.abs(delta_new - delta)) < 1e-12:
                break
            delta = delta_new

        return delta.numpy()

    def compute_gmm_objective(self, sigma: float) -> float:
        """
        Calculates the GMM objective function value for a given sigma.
        Concentrates out linear parameters (beta) via 2SLS.
        """
        if sigma <= 0.01 or sigma > 5.0:
            return 1e10

        delta = self.solve_delta(sigma).reshape(-1)

        # 2SLS Projection: X_hat = P_Z * X
        # beta = (X_hat' X)^-1 X_hat' delta
        Pz = self.Z @ self.inv_ZZ @ self.Z.T
        X_hat = Pz @ self.X

        try:
            beta = np.linalg.solve(X_hat.T @ self.X, X_hat.T @ delta)
        except np.linalg.LinAlgError:
            return 1e10

        # Structural error
        xi = delta - self.X @ beta

        # GMM Objective: xi' Z (Z'Z)^-1 Z' xi
        val = (self.Z.T @ xi).T @ self.inv_ZZ @ (self.Z.T @ xi)

        # Store for retrieval
        self.last_beta = beta
        self.last_xi = xi
        return val

In [ ]:
# %% [Cell 3] Bayesian Shrinkage MCMC

class BayesianShrinkageEstimator:
    """
    Implements the Bayesian Shrinkage estimator using MCMC (Gibbs + MH).
    Uses a Spike-and-Slab prior to enforce sparsity on structural errors.
    """

    def __init__(self, dataset: ChoiceDataset, T: int, J_real: int):
        self.T = T
        self.J_real = J_real
        self.J_total = J_real + 1

        self.p, self.w, self.counts = self._extract_data(dataset)
        # X for Bayesian model: [p, w] (Intercept handled by xi_bar)
        self.X_cov = np.stack([self.p, self.w], axis=2)

        # Priors (from paper)
        self.tau0_sq = 1e-3  # Spike variance
        self.tau1_sq = 1.0   # Slab variance
        self.a_phi = 1.0
        self.b_phi = 1.0

        self.nu_i = np.random.normal(0, 1, (200, 1))

    def _extract_data(self, dataset):
        feat_array = dataset.items_features_by_choice[0]
        raw_feats = np.array(feat_array).reshape(self.T, -1, self.J_total, 2)
        p = raw_feats[:, 0, 1:, 0].astype(np.float64)
        w = raw_feats[:, 0, 1:, 1].astype(np.float64)

        choices_mat = dataset.choices.reshape(self.T, -1)
        counts = np.array([np.bincount(choices_mat[t], minlength=self.J_total) for t in range(self.T)])
        return p, w, counts

    def _compute_choice_probs(self, beta, sigma, xi_bar, eta):
        """Calculates market shares using the Logit integral."""
        # Mean utility
        xb = np.dot(self.X_cov, beta)
        delta = xb + xi_bar[:, np.newaxis] + eta

        # Random coefficients
        p = self.X_cov[..., 0]
        mu = sigma * self.nu_i[np.newaxis, :, :] * p[:, np.newaxis, :]

        # Probabilities
        exp_u = np.exp(delta[:, np.newaxis, :] + mu)
        sum_exp = 1.0 + np.sum(exp_u, axis=2, keepdims=True)
        probs = np.mean(exp_u / sum_exp, axis=1)

        # Outside option share
        s0 = 1.0 - np.sum(probs, axis=1, keepdims=True)
        return np.column_stack([s0, probs])

    def _log_likelihood(self, probs):
        return np.sum(self.counts * np.log(np.maximum(probs, 1e-10)))

    def fit(self, n_iter=2000, burn_in=1000):
        """
        Executes the MCMC sampling loop (Cold Start).
        """
        # Initialization (Random Cold Start)
        beta = np.array([-1.0, 0.5])
        sigma = 1.5
        xi_bar = np.full(self.T, -1.0)
        phi = np.full(self.T, 0.5)

        # Initialize eta randomly to avoid "zero-trap" deadlock
        eta = np.random.normal(0, 1.0, (self.T, self.J_real))
        gamma = (np.abs(eta) > 0.05).astype(float)

        curr_probs = self._compute_choice_probs(beta, sigma, xi_bar, eta)
        curr_ll = self._log_likelihood(curr_probs)

        store_params = []
        store_eta_total = []
        store_gamma = []

        # MCMC Loop
        for it in range(n_iter):
            # 1. Update Beta (Slope)
            beta, curr_ll, curr_probs = self._mh_step_beta(beta, sigma, xi_bar, eta, curr_ll, curr_probs)

            # 2. Update Sigma (Random Coefficient)
            sigma, curr_ll, curr_probs = self._mh_step_sigma(beta, sigma, xi_bar, eta, curr_ll, curr_probs)

            # 3. Update Xi_bar (Market Intercepts)
            xi_bar, curr_ll, curr_probs = self._mh_step_xi_bar(beta, sigma, xi_bar, eta, curr_ll, curr_probs)

            # 4. Update Eta (Structural Error)
            eta, curr_ll, curr_probs = self._mh_step_eta(beta, sigma, xi_bar, eta, gamma, curr_ll, curr_probs)

            # 5. Update Gamma (Spike Indicator)
            gamma = self._gibbs_step_gamma(eta, phi)

            # 6. Update Phi (Sparsity Probability)
            phi = self._gibbs_step_phi(gamma)

            # Storage with thinning
            if it >= burn_in and it % 2 == 0:
                total_xi = xi_bar[:, None] + eta
                est_int = np.mean(total_xi)
                store_params.append([est_int, beta[0], beta[1], sigma])
                store_eta_total.append(total_xi)
                store_gamma.append(gamma)

        return np.mean(store_params, 0), np.mean(store_eta_total, 0), np.mean(store_gamma, 0)

    # --- MCMC Helper Methods ---

    def _mh_step_beta(self, beta, sigma, xi_bar, eta, curr_ll, curr_probs):
        prop_beta = beta + np.random.normal(0, 0.02, 2)
        prop_probs = self._compute_choice_probs(prop_beta, sigma, xi_bar, eta)
        prop_ll = self._log_likelihood(prop_probs)

        prior_diff = (-0.5 * np.sum(prop_beta**2) / 10.0) - (-0.5 * np.sum(beta**2) / 10.0)

        if np.log(np.random.rand()) < (prop_ll - curr_ll + prior_diff):
            return prop_beta, prop_ll, prop_probs
        return beta, curr_ll, curr_probs

    def _mh_step_sigma(self, beta, sigma, xi_bar, eta, curr_ll, curr_probs):
        prop_sigma = sigma + np.random.normal(0, 0.05)
        if prop_sigma <= 0.05:
            return sigma, curr_ll, curr_probs

        prop_probs = self._compute_choice_probs(beta, prop_sigma, xi_bar, eta)
        prop_ll = self._log_likelihood(prop_probs)

        def log_prior(s): return -np.log(s) - (np.log(s)**2)

        if np.log(np.random.rand()) < (prop_ll - curr_ll + log_prior(prop_sigma) - log_prior(sigma)):
            return prop_sigma, prop_ll, prop_probs
        return sigma, curr_ll, curr_probs

    def _mh_step_xi_bar(self, beta, sigma, xi_bar, eta, curr_ll, curr_probs):
        prop_xi_bar = xi_bar + np.random.normal(0, 0.05, self.T)
        prop_probs = self._compute_choice_probs(beta, sigma, prop_xi_bar, eta)
        prop_ll = self._log_likelihood(prop_probs)

        prior_diff = (-0.5 * np.sum(prop_xi_bar**2) / 10.0) - (-0.5 * np.sum(xi_bar**2) / 10.0)

        if np.log(np.random.rand()) < (prop_ll - curr_ll + prior_diff):
            return prop_xi_bar, prop_ll, prop_probs
        return xi_bar, curr_ll, curr_probs

    def _mh_step_eta(self, beta, sigma, xi_bar, eta, gamma, curr_ll, curr_probs):
        prop_eta = eta + np.random.normal(0, 0.1, (self.T, self.J_real))
        var = np.where(gamma == 1, self.tau1_sq, self.tau0_sq)

        prior_curr = -0.5 * np.sum(eta**2 / var)
        prior_prop = -0.5 * np.sum(prop_eta**2 / var)

        prop_probs = self._compute_choice_probs(beta, sigma, xi_bar, prop_eta)
        prop_ll = self._log_likelihood(prop_probs)

        if np.log(np.random.rand()) < (prop_ll - curr_ll + prior_prop - prior_curr):
            return prop_eta, prop_ll, prop_probs
        return eta, curr_ll, curr_probs

    def _gibbs_step_gamma(self, eta, phi):
        d1 = norm.pdf(eta, 0, np.sqrt(self.tau1_sq)) * phi[:, None]
        d0 = norm.pdf(eta, 0, np.sqrt(self.tau0_sq)) * (1 - phi[:, None])
        prob_1 = d1 / (d1 + d0 + 1e-12)
        return (np.random.rand(*eta.shape) < prob_1).astype(float)

    def _gibbs_step_phi(self, gamma):
        sum_g = gamma.sum(axis=1)
        return np.random.beta(self.a_phi + sum_g, self.b_phi + (self.J_real - sum_g))

In [ ]:
# %% [Cell 4] Main

class MetricsCalculator:
    TRUE_PARAMS = np.array([-1.0, -1.0, 0.5, 1.5])

    @staticmethod
    def compute_row(results, dgp_type, method_type):
        res_matrix = np.array(results)
        param_bias = np.mean(res_matrix[:, :4] - MetricsCalculator.TRUE_PARAMS, axis=0)
        param_sd = np.std(res_matrix[:, :4], axis=0)
        xi_bias = np.mean(res_matrix[:, 4])
        xi_sd = np.std(res_matrix[:, 4])
        row_bias = list(param_bias) + [xi_bias]
        row_sd = list(param_sd) + [xi_sd]
        if method_type == 'shrink' and dgp_type in [1, 2]:
            row_bias.append(np.mean(res_matrix[:, 5]))
            row_sd.append(np.mean(res_matrix[:, 6]))
        else:
            row_bias.append(np.nan)
            row_sd.append(np.nan)
        return row_bias, row_sd

def run_full_factorial_replication():
    """Main execution loop for the full factorial experiment."""

    # Experiment Configuration
    #T_list = [25, 100]
    T_list = [100]
    #J_list = [5, 15]
    J_list = [15]
    dgp_list = [1, 2, 3, 4]
    n_reps = 50

    total_scenarios = len(T_list) * len(J_list) * len(dgp_list)
    print(f"Starting Full Replication: {total_scenarios} scenarios, {n_reps} reps each.")

    for T in T_list:
        for J in J_list:
            print(f"\n{'='*60}")
            print(f"SCENARIO: Markets (T)={T}, Products (J)={J}")
            print(f"{'='*60}")

            for dgp in dgp_list:
                print(f"\n>>> Running DGP {dgp} ...")

                # Containers for results
                results = {'blp_cost': [], 'blp_nocost': [], 'shrink': []}

                # FIXED: Explicitly call tqdm.tqdm to resolve TypeError
                iterator = tqdm_module.tqdm(range(n_reps), leave=False, desc=f"Sim T{T}J{J}D{dgp}")

                for r in iterator:
                    # 1. Generate Data
                    # Unique seed for every combination
                    seed = (T * 100000) + (J * 1000) + (dgp * 100) + r
                    sim = SimulationExperiment(T, 1000, J, dgp, seed)
                    dataset = sim.run()
                    true_eta_dev = dataset.true_eta.reshape(-1)

                    # 2. BLP (Cost IV)
                    blp1 = BLPEstimator(dataset, T, J, 'cost_iv')
                    opt1 = minimize_scalar(blp1.compute_gmm_objective, bounds=(0.1, 4.0), method='bounded')
                    xi_b1 = np.mean(np.abs(blp1.last_xi - true_eta_dev))
                    results['blp_cost'].append([*blp1.last_beta, opt1.x, xi_b1])

                    # 3. BLP (No Cost IV)
                    blp2 = BLPEstimator(dataset, T, J, 'no_cost_iv')
                    opt2 = minimize_scalar(blp2.compute_gmm_objective, bounds=(0.1, 4.0), method='bounded')
                    xi_b2 = np.mean(np.abs(blp2.last_xi - true_eta_dev))
                    results['blp_nocost'].append([*blp2.last_beta, opt2.x, xi_b2])

                    # 4. Bayesian Shrinkage
                    mcmc = BayesianShrinkageEstimator(dataset, T, J)
                    est_p, est_eta_tot, est_gamma = mcmc.fit(n_iter=5000, burn_in=2500)

                    est_eta_dev = est_eta_tot - np.mean(est_eta_tot)
                    xi_b3 = np.mean(np.abs(est_eta_dev - dataset.true_eta))

                    # Prob Calculation logic
                    res_row = [*est_p, xi_b3]
                    if dgp in [1, 2]:
                        flat_g = est_gamma.flatten()
                        is_nz = np.abs(true_eta_dev) > 0.1
                        p1 = np.mean(flat_g[is_nz]) if is_nz.sum() > 0 else 0
                        p2 = np.mean(flat_g[~is_nz]) if (~is_nz).sum() > 0 else 0
                        res_row.extend([p1, p2])
                    else:
                        res_row.extend([np.nan, np.nan])

                    results['shrink'].append(res_row)

                    # Cleanup
                    del dataset, sim, blp1, blp2, mcmc
                    gc.collect()

                # --- Aggregate and Display Results ---
                rows = []
                index_names = []

                # Process BLP Cost
                b, s = MetricsCalculator.compute_row(results['blp_cost'], dgp, 'blp')
                rows.extend([b, s])
                index_names.extend(['BLP(Cost)-Bias', 'BLP(Cost)-SD'])

                # Process BLP No Cost
                b, s = MetricsCalculator.compute_row(results['blp_nocost'], dgp, 'blp')
                rows.extend([b, s])
                index_names.extend(['BLP(NoCost)-Bias', 'BLP(NoCost)-SD'])

                # Process Shrinkage
                b, s = MetricsCalculator.compute_row(results['shrink'], dgp, 'shrink')
                rows.extend([b, s])
                index_names.extend(['Shrinkage-Bias', 'Shrinkage-SD'])

                df = pd.DataFrame(rows, columns=['Int', 'Bp', 'Bw', 'Sig', 'Xi', 'Prob'], index=index_names)
                print(f"Results for T={T}, J={J}, DGP={dgp}:")
                print(df.round(3).fillna('-'))

In [ ]:
# %% [Cell 5] Unittest

class TestBLPPipeline(unittest.TestCase):
    def test_simulation_shape(self):
        T, N, J = 5, 100, 3
        sim = SimulationExperiment(T, N, J, dgp_type=1, seed=42)
        dataset = sim.run()
        feats = dataset.items_features_by_choice[0]
        self.assertEqual(feats.shape, (T * N, J + 1, 2))

    def test_blp_estimator_run(self):
        T, N, J = 5, 100, 3
        sim = SimulationExperiment(T, N, J, dgp_type=1, seed=42)
        dataset = sim.run()
        blp = BLPEstimator(dataset, T, J, 'no_cost_iv')
        val = blp.compute_gmm_objective(1.5)
        self.assertIsInstance(val, float)

    def test_bayesian_mcmc_step(self):
        T, N, J = 5, 100, 3
        sim = SimulationExperiment(T, N, J, dgp_type=1, seed=42)
        dataset = sim.run()
        # FIXED: Correct Class Name
        mcmc = BayesianShrinkageEstimator(dataset, T, J)
        p, eta, gamma = mcmc.fit(n_iter=10, burn_in=5)
        self.assertEqual(len(p), 4)
        self.assertEqual(eta.shape, (T, J))
        self.assertEqual(gamma.shape, (T, J))

if __name__ == '__main__':
    # 1. Run Unit Tests
    print("Running Unit Tests...")
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

    # 2. Run Main Experiment
    print("\nRunning Main Experiment...")
    run_full_factorial_replication()

Running Unit Tests...


...
----------------------------------------------------------------------
Ran 3 tests in 1.507s

OK



Running Main Experiment...
Starting Full Replication: 4 scenarios, 50 reps each.

SCENARIO: Markets (T)=100, Products (J)=15

>>> Running DGP 1 ...


Sim T100J15D1:  74%|███████▍  | 37/50 [3:02:28<1:03:58, 295.29s/it]